## 1. Feature extraction for hybrid retrieval
Chunk embeddings are computed using the [Multilingual E5 Base](https://huggingface.co/intfloat/multilingual-e5-base) model for semantic retrieval.

Words are extracted from chunks for keyword-based retrieval.

In [ ]:
# Install packages for generating embeddings and a BM25 index,
# and for accessing a chroma database
%pip install -U transformers
%pip install -U chromadb
%pip install -U rank_bm25

In [ ]:
# Feature extraction functions for dense retrieval (embeddings)

import torch

def mean_pool(last_hidden_states, attention_mask):
    """
    Computes a sequence embedding by mean-pooling the embeddings of all
    non-padding tokens.

    Args:
        last_hidden_states: Tensor of shape (batch_size, seq_len, hidden_dim).
        attention_mask: Tensor of shape (batch_size, seq_len).

    Returns:
        A tensor of shape (batch_size, hidden_dim).
    """

    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)

    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def get_embedding(text, tokenizer, model):
    """
    Generates a text embedding using a transformer encoder and mean pooling.

    Args:
        text: Input text.
        tokenizer: Tokenizer associated with the model.
        model: Transformer encoder model.

    Returns:
        A tensor containing the text embedding.
    """

    inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**inputs)
    embedding = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])

    return embedding.squeeze()

In [ ]:
# Feature extraction class for sparse retrieval (BM25)

import json
import math
import re

from rank_bm25 import BM25Okapi

class BM25Log1(BM25Okapi):
    """
    BM25 variant with strictly non-negative IDF values.

    This class extends the standard BM25Okapi implementation by replacing the
    original IDF formulation with a smoothed logarithmic variant:

        IDF(t) = log(1 + (N - df(t) + 0.5) / (df(t) + 0.5))

    This transformation guarantees:
        - IDF(t) >= 0 for all terms
        - monotonic decrease of IDF with term frequency across documents
        - elimination of negative IDF values present in classical BM25

    Rationale:
        In standard BM25, IDF can become negative when a term appears in more
        than half of the documents. While theoretically consistent with the
        probabilistic interpretation of BM25, negative IDF values may lead to
        unintuitive behavior in practical retrieval systems, where very common
        terms can penalize document scores.

        This variant removes such penalization while preserving the relative
        weighting behavior of BM25 for rare and moderately frequent terms.
    """

    def __init__(self, chunk_terms, chunk_ids, **kwargs):
        super().__init__(chunk_terms, **kwargs)

        self.chunk_ids = chunk_ids

        df = {}
        for doc in self.doc_freqs:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1

        N = self.corpus_size
        for term, dfi in df.items():
            self.idf[term] = math.log(1 + (N - dfi + 0.5) / (dfi + 0.5))

    @staticmethod
    def extract_terms(text):
        """
        Extracts lowercase alphabetic terms from a text for BM25 indexing
        and keyword-based retrieval.

        Args:
            text: Input text.

        Returns:
            A list of normalized terms.
        """

        return re.findall(r"\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b", text.lower())


    def save(self, path):
        """Serialize BM25 index to JSON"""
        data = {
            "chunk_ids": self.chunk_ids,
            "idf": {k: float(v) for k, v in self.idf.items()},
            "doc_freqs": self.doc_freqs,
            "doc_len": self.doc_len,
            "avgdl": self.avgdl,
            "corpus_size": self.corpus_size,
            "k1": self.k1,
            "b": self.b
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path):
        """Load BM25 index from JSON without recomputing statistics."""
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        bm25 = cls.__new__(cls)
        bm25.chunk_ids = data["chunk_ids"]
        bm25.idf = {k: float(v) for k, v in data["idf"].items()}
        bm25.doc_freqs = data["doc_freqs"]
        bm25.doc_len = data["doc_len"]
        bm25.avgdl = data["avgdl"]
        bm25.corpus_size = data["corpus_size"]
        bm25.k1 = data["k1"]
        bm25.b = data["b"]

        return bm25

In [ ]:
# Hybrid (dense + sparse) feature extraction

import uuid
from datetime import datetime

from google.colab import drive
from transformers import AutoTokenizer, AutoModel

# Mount Google Drive for file access
drive.mount("/content/drive", force_remount=False)

# Model used for embedding generation: Multilingual E5 Base
model_name = "intfloat/multilingual-e5-base"

# Load the model and tokenizer from Hugging Face Hub
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Feature extraction metadata
feature_extraction_process = {
        "process_id": str(uuid.uuid4()),
        "timestamp": str(int(datetime.now().timestamp())),
        "model": model_name
    }

# Load the chunks file into memory
CHUNK_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"
chunks_file_name = "chunks_table_1781248570.json" # @param {type:"string"}
chunks_file_path = f"{CHUNK_DIR}/{chunks_file_name}"
with open(chunks_file_path, encoding="utf-8") as f:
    input = json.load(f)

# Enrich chunk info with embeddings and terms
chunking_process = input["chunking_process"]
chunk_info_list_in = input["chunks"]
chunk_info_list_out = []
chunk_terms = []
chunk_ids = []
for chunk_info in chunk_info_list_in:
    embedding = get_embedding(chunk_info["chunk_text"], tokenizer, model)
    terms = BM25Log1.extract_terms(chunk_info["chunk_text"])
    chunk_info.update({
        "embedding": embedding.detach().cpu().tolist(),
        "terms": terms})
    chunk_info_list_out.append(chunk_info)
    chunk_terms.append(terms)
    chunk_ids.append(chunk_info["chunk_id"])

# Output to be serialized
output = {
    "chunking_process": chunking_process,
    "feature_extraction_process": feature_extraction_process,
    "chunks": chunk_info_list_out
    }

# Save chunks to a JSON file
with open(chunks_file_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

## 2. Indexing for hybrid retrieval
Chunks are stored in a Chroma database, which builds a HNSW index for fast semantic (_dense_) retrieval.

A BM25 ranking index is built in memory over chunks for fast keyword-based (_sparse_) retrieval, using a non-negative IDF variant to ensure very frequent terms do not reduce relevance scores below the level expected after filtering common terms.

In [ ]:
# Store chunks, embeddings, and metadata in ChromaDB and persists the
# BM25 index to a file

import chromadb

tst = str(int(datetime.now().timestamp()))

DB_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"
db_file_name = f"chroma_db_{tst}"
db_file_path = f"{DB_DIR}/{db_file_name}"

client = chromadb.PersistentClient(path=db_file_path)

# Delete collection if it exists (to avoid duplicate inserts on re-runs)
collection_name = "construction_site_visit_reports"

try:
    client.delete_collection(collection_name)
except Exception:
    pass

# Create a collection with cosine distance (HNSW index internally)
collection = client.create_collection(
    name=collection_name,
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

# Add chunks, their embeddings, and metadata to the vector database
collection.add(
    ids=[str(chunk_info["chunk_id"]) for chunk_info in chunk_info_list_out],
    documents=[chunk_info["chunk_text"] for chunk_info in chunk_info_list_out],
    embeddings=[chunk_info["embedding"] for chunk_info in chunk_info_list_out],
    metadatas=[{"docx_id": chunk_info["docx_id"],
                "chunk_id": chunk_info["chunk_id"],
                "chunk_index": chunk_info["chunk_index"]}
                for chunk_info in chunk_info_list_out]
)

# Build BM25 index from the corpus of chunk terms
bm25 = BM25Log1(chunk_terms, chunk_ids)

# Persist BM25 index into a file (uses the same tst as the database file in the
# file name)
BM25_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"
bm25_file_name = f"bm25_{tst}.json"
bm25_file_path = f"{BM25_DIR}/{bm25_file_name}"
bm25.save(bm25_file_path)

print(f"{collection_name = }")
print(f"Stored {collection.count()} records in ChromaDB.")
print(f"Stored {len(chunk_terms)} records in bm25.")
